# Step 1 Master Orchestrator: AD Research Pipeline
This notebook serves as the master execution client for Step 1 mapping Clinical Phenotypes, Stratification, and Cell-type Deconvolution.
It is completely standalone and designed to run natively in Google Colab.


In [1]:
# ==== Environment Setup (Google Colab) ====
# This cell clones the repository and installs all dependencies automatically
import os
import sys

# Check if we are in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Detected Google Colab Environment.")
except ImportError:
    IN_COLAB = False
    print("Detected Local Environment.")

if IN_COLAB:
    # Clone the repository if it doesn't already exist
    if not os.path.exists('ad_gnn_pipeline'):
        !git clone https://github.com/SalmanSattar24/ad_gnn_pipeline.git
    
    # Change directory so imports work correctly
    if os.path.basename(os.getcwd()) != 'ad_gnn_pipeline':
        os.chdir('ad_gnn_pipeline')
        
    print("Installing dependencies from requirements.txt...")
    !pip install -q -r requirements.txt
    print("Setup complete. Current Directory:", os.getcwd())
else:
    print("Running locally. Ensure you are in the ad_gnn_pipeline root directory or 'notebooks' folder.")


Detected Local Environment.
Running locally. Ensure you are in the ad_gnn_pipeline root directory or 'notebooks' folder.


In [2]:
import sys
import os
import logging

# Ensure the kernel path is rooted properly to import src modules
current_dir = os.path.basename(os.getcwd())
if current_dir == 'notebooks':
    sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
    DATA_DIR = '../data'
    RESULTS_DIR = '../results'
else:
    # In Colab, we should be at the root of the repo
    sys.path.insert(0, os.path.abspath('src'))
    DATA_DIR = 'data'
    RESULTS_DIR = 'results'

try:
    from step1.step_1a_load_preprocess import main as run_step_1a
    from step1.step_1b_deconvolution import main as run_step_1b
    from step1.step_1c_pseudotime import main as run_step_1c
    from step1.step_1c2_asymad import main as run_step_1c2
    from step1.step_1d_nmf_clustering import main as run_step_1d
    from step1.step_1e_subtype_validation import main as run_step_1e
    print("Successfully imported pipeline modules!")
except ImportError as e:
    print(f"Failed to import modules: {e}")
    print(f"Current Sys Path: {sys.path}")

# Setup basic logging to see the pipeline's info messages natively in Colab outputs
logging.basicConfig(level=logging.INFO, format='%(name)s - %(message)s', force=True)

# Configuration 
TEST_MODE = False  # Set to True for a quick 50-patient mock run execution


Successfully imported pipeline modules!


In [3]:
# ==== Step 1A: Load and Preprocess ====
print("Starting Step 1A: Data Loading & Preprocessing...")
# If raw data doesn't exist, this will automatically generate synthetic data.
res_1a = run_step_1a(data_dir=DATA_DIR, results_dir=RESULTS_DIR, test_mode=TEST_MODE)
print(f"Outcome: {res_1a}")


Step1A - ======================================================================


Step1A - STEP 1A: Data Loading & Preprocessing


Step1A - ======================================================================


Starting Step 1A: Data Loading & Preprocessing...


root - [SETUP] Generated synthetic proteomics: (180, 5000)


...

root -   Saved: qc_report.png


Step1A - ======================================================================


Step1A - STEP 1A COMPLETE


Step1A - ======================================================================


Outcome: {'n_samples': 180, 'n_proteins': 5000, 'missing_pct': np.float64(0.0), 'status': 'PASS'}


In [4]:
# ==== Step 1B: Cell-Type Deconvolution ====
print("\nStarting Step 1B: Cell-Type Deconvolution...")
res_1b = run_step_1b(data_dir=DATA_DIR, results_dir=RESULTS_DIR, skip_deconvolution=False, test_mode=TEST_MODE)
print(f"Outcome: {res_1b}")


Step1B - ======================================================================


Step1B - STEP 1B: Cell-Type Deconvolution


Step1B - ======================================================================


Step1B - [1/5] Loading reference...



Starting Step 1B: Cell-Type Deconvolution...


...

root -   Saved: cell_type_proportions.png


Step1B - ======================================================================


Step1B - STEP 1B COMPLETE


Step1B - ======================================================================


Outcome: {'n_cell_types': 6, 'n_patients': 180, 'status': 'PASS'}


In [5]:
# ==== Step 1C: Disease Pseudotime Computation ====
print("\nStarting Step 1C: Pseudotime Computation...")
res_1c = run_step_1c(data_dir=DATA_DIR, results_dir=RESULTS_DIR, test_mode=TEST_MODE)
print(f"Outcome: {res_1c}")


Step1C - ======================================================================


Step1C - STEP 1C: Disease Pseudotime


Step1C - ======================================================================


Step1C - [1/6] Loading data...



Starting Step 1C: Pseudotime Computation...


...

root -   Saved: pseudotime_umap.png


Step1C - ======================================================================


Step1C - STEP 1C COMPLETE


Step1C - ======================================================================


Outcome: {'pseudotime_min': 0.0, 'pseudotime_max': 1.0, 'pseudotime_mean': 0.40078243613243103, 'valid': False, 'status': 'WARNING'}


In [6]:
# ==== Step 1C2 (Novel Addition): AsymAD / Cognitive Resilience Inversion ====
print("\nStarting Step 1C2: AsymAD Definition...")
res_1c2 = run_step_1c2(data_dir=DATA_DIR, results_dir=RESULTS_DIR, test_mode=TEST_MODE)
print(f"Outcome: {res_1c2}")


Step1C2_AsymAD - ======================================================================


Step1C2_AsymAD - STEP 1C (Addition): Cognitive Resilience Inversion Analysis


Step1C2_AsymAD - ======================================================================


Step1C2_AsymAD - [1/3] Loaded master patient table: (180, 17)


Step1C2_AsymAD - [2/3] Classifying patients...


...

root -   Updated and saved: master_patient_table.csv (added 'patient_class')


Step1C2_AsymAD - ======================================================================


Step1C2_AsymAD - STEP 1C (Addition) COMPLETE


Step1C2_AsymAD - ======================================================================



Starting Step 1C2: AsymAD Definition...
Outcome: {'asymad_count': 24, 'ad_count': 76, 'control_count': 80, 'status': 'PASS'}


In [7]:
# ==== Step 1D: Molecular Subtype Clustering (NMF) ====
print("\nStarting Step 1D: NMF Clustering...")
res_1d = run_step_1d(data_dir=DATA_DIR, results_dir=RESULTS_DIR, n_subtypes=None, test_mode=TEST_MODE)
print(f"Outcome: {res_1d}")


Step1D - ======================================================================


Step1D - STEP 1D: NMF Consensus Clustering


Step1D - ======================================================================


Step1D - [1/5] Loading data...


root -   Loaded 76 AD/MCI patients


...

root -   Saved: subtype_cluster_sizes.png


Step1D - ======================================================================


Step1D - STEP 1D COMPLETE


Step1D - ======================================================================


Outcome: {'n_subtypes': 2, 'subtype_sizes': {'ST1': 53, 'ST2': 23}, 'status': 'PASS'}


In [8]:
# ==== Step 1E: Subtype Validation & Clinical Analysis ====
print("\nStarting Step 1E: Subtype Validation...")
res_1e = run_step_1e(data_dir=DATA_DIR, results_dir=RESULTS_DIR, test_mode=TEST_MODE)
print(f"Outcome: {res_1e}")


Step1E - ======================================================================


Step1E - STEP 1E: Subtype Validation and Clinical Mapping


Step1E - ======================================================================


Step1E - [1/7] Loading data...


root -   Loaded master table: (180, 19)


...

root -   Saved: step1_summary.txt


Step1E - ======================================================================


Step1E - STEP 1E COMPLETE


Step1E - ======================================================================


Outcome: {'n_subtypes': 2, 'n_control': 104, 'subtype_list': ['ST1', 'ST2'], 'status': 'PASS'}


In [9]:
print(f"\nâœ¨ Step 1 Pipeline Completed Successfully! âœ¨")
print(f"Outputs are saved in {DATA_DIR}/processed/ and {RESULTS_DIR}/step1/")



âœ¨ Step 1 Pipeline Completed Successfully! âœ¨
Outputs are saved in data/processed/ and results/step1/
